# 05 · Atmospheric Transport & Source Attribution

Simulating atmospheric back-trajectories and attributing downwind pollution (e.g. Delhi NCR) to upwind crop residue burning and active fire detections.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from isro_aqi.synthetic import SyntheticConfig, generate_all
from isro_aqi.hcho.transport import compute_back_trajectory, correlate_trajectory_with_fires


### 1. Compute 48-hour Back-Trajectory from Delhi


In [ ]:
cfg = SyntheticConfig(resolution_deg=0.5, n_days=15)
data = generate_all(cfg)

delhi_lon, delhi_lat = 77.10, 28.65
u10 = data["stack"]["u10"]
v10 = data["stack"]["v10"]

traj = compute_back_trajectory(
    receptor_lon=delhi_lon,
    receptor_lat=delhi_lat,
    target_time=data["stack"]["time"].values[5],
    u_da=u10,
    v_da=v10,
    hours=48,
    dt_hours=1.0
)
print(f"Trajectory length: {len(traj)} steps")
traj.head()


### 2. Intersect Trajectory with Active Fire Pixels


In [ ]:
fires = data["fires"]
matched = correlate_trajectory_with_fires(traj, fires, max_dist_km=50)

plt.figure(figsize=(9, 7))
plt.scatter(fires["longitude"], fires["latitude"], c="orangered", s=fires["frp"]/4, alpha=0.5, label="Fire Pixels (FRP)")
plt.plot(traj["lon"], traj["lat"], color="black", linewidth=2.5, linestyle="--", label="48h Back-Trajectory")
plt.scatter([delhi_lon], [delhi_lat], color="blue", s=100, zorder=5, label="Delhi Receptor")

plt.title("Punjab–Haryana Fires & Back-Trajectory to Delhi")
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.show()
